# Expected Value

In building up tools to support decision making under uncertainty, we first need to establish the idea of expected value. This notion will allow us to deal with a range of different states of the world, and is intuitively useful to grasp before the idea of a utility function, which will rely on very similar mathematical machinery.

In [1]:
import numpy as np
import pymc as pm
import matplotlib.pyplot as plt 
from typing import List, Dict, Set, Callable

## A Cash Giveaway

When we have clearly *and deterministically* defined outcomes, choices are easy to make. Suppose someone offers you a choice between to options, $X$ and $Y$:

+ $X$: I'll give you $5
+ $Y$: I'll give you $10

The payoffs are quite clear here, and in the absence of any other considerations, the "rational" choice is clear. $10 is unambiguously more than $5, so option $Y$ should be the preferred one. Even in a slightly more complicated example, the same logic holds. Suppose a rather **un**saavy issuer offers you debt options:

+ $X$: A  10-year $1,000 [bullet bond](https://www.investopedia.com/terms/b/bulletbond.asp#toc-what-is-a-bullet-bond) with an interest rate of 10% for $200.
+ $Y$: A 20-year $10,000 [bullet bond](https://www.investopedia.com/terms/b/bulletbond.asp#toc-what-is-a-bullet-bond) with at an interest rate of 13% for $500.

Given all the parameters, it is a little more complicated to work out, but we can deterministically compute the net present value for both options.

In [8]:
def npv_bb(interest: float, periods: int = 10, principal: float = 1000) -> float:
    discount: float = float(1) / (1 + interest) ** periods
    return principal * discount

npv_bb_x: float = npv_bb(interest=0.1, periods=10, principal=1000)
npv_bb_y: float = npv_bb(interest=0.13, periods=20, principal=10000)
npv_bb_x_cost: float = 200.
npv_bb_y_cost: float = 500.
npv_bb_x_payout: float = npv_bb_x - npv_bb_x_cost
npv_bb_y_payout: float = npv_bb_y - npv_bb_y_cost
print(f"Bullet Bond X Payout: {npv_bb_x_payout}")
print(f"Bullet Bond Y Payout: {npv_bb_y_payout}")

Bullet Bond X Payout: 185.5432894295314
Bullet Bond Y Payout: 367.82294851679274


Again, the bigger number wins. In this case, $Y$ is still the preferable option because it is a bigger number in our assumed state of the world. The question is, what happens if we allow for fundamental parameters to vary? 

Suppose we alter our example a bit. We aren't choosing between two debt products, but instead are rendering a decision about opportunity cost. We can either invest in the debt of the issuer, or invest in equities via the S&P Index. Between 2019 and 2024, the S&P rose by 82%. This implies an average annual growth rate of about 12.7%. Our choices are now the following:

+ $X$: A 20-year $10,000 [bullet bond](https://www.investopedia.com/terms/b/bulletbond.asp#toc-what-is-a-bullet-bond) with an interest rate of 13% for $500.
+ $Y$: Investing $200 in the S&P over the same period.

The problem now is that we need to question how much faith we have in the stability of the S&P growth rate. We cannot just blindly accept the growth rate from the last 5 years, because the next 5 could look different. 

Suppose there are two very different states of the world, predicated on different scenarios. Maybe current policy will boost exports and jump start domestic manufacturing, in which case we *expect* to see a growth rate of 13%. Maybe current policy will create uncertainty for consumers and businesses, reducing the amount of capital available in the private sector, corresponding to an expectation of 2.5%. We are unsure of which scenario will occur, but (for now) we are certain one will occur. To operationalize this uncertainty, we need a probability distribution. Let us presume $P(S_1)$ is 0.2 and $P(S_2)$ is 0.8. In other words, if $p$ can take values in $[0, 1]$ then we are saying $p$ is 0.2 and represents the high equity growth scenario. Correspondingly, the low equity growth scenario occurs with probability $1-p$ (i.e. 0.8):

$$
i(S) = 
\begin{cases}
  13\% & \text{if } p < 0.2 \\
  2.5\% & \text{otherwise}
\end{cases}
$$

The expected value of the S&P investment is just the size of the payout for each scenario times the probability of that scenario occuring:

$$ E(\text{payout}) = \Sigma_i^n p_{s_i} S_i = (0.2)[(1.13)^{10}(200)] + (0.8)[(1.025)^{10}(200)] = ??$$

In [15]:
s_and_p: Callable[[float, float, int], float] = lambda principal, interest, periods: ((1+interest)**periods) * principal
print(0.2 * s_and_p(200, 0.13, 10) + 0.8 * s_and_p(200, 0.025, 10))

340.59622266830587


Now we are in a position to do some interesting things. First, we note that our expected value is actually below the yield from the debt investment, so *if we have pegged $p$ appropriately, we should choose the debt investment. However, $p$ is a random variable. It could be higher or lower in reality. So, even in the two scenario case, we might want to know how likely we are to get $p = 0.2$. 